# OTEL Metrics — Silver Pipeline

Spark Declarative Pipeline that creates silver tables from raw OTEL metrics data.

| Table | Type | Description |
| --- | --- | --- |
| `node_compute` | Streaming | Node CPU & memory metrics (long format) |
| `node_filesystem` | Materialized View | Node filesystem with pivoted available/capacity/usage |
| `node_network` | Streaming | Node network IO with interface and direction |
| `pod_compute` | Streaming | Pod CPU & memory metrics with full K8s hierarchy |
| `pod_filesystem` | Materialized View | Pod filesystem with pivoted available/capacity/usage |
| `pod_network` | Streaming | Pod network IO with interface and direction |
| `container_resource_spec` | Materialized View | Latest CPU/mem requests + limits per container (cluster receiver) |
| `node_capacity_estimated` | Materialized View | Estimated node CPU & memory capacity derived from usage/utilization |
| `bin_packing_pod_efficiency` | Materialized View | Per-pod request vs actual-usage analysis for bin-packing |
| `bin_packing_node_summary` | Materialized View | Per-node aggregate of pod requests, usage, headroom |

In [ ]:
import dlt
from pyspark.sql import functions as F

source_catalog = spark.conf.get("source_catalog")
source_schema = spark.conf.get("source_schema")
table_prefix = spark.conf.get("table_prefix")
source_table = f"{source_catalog}.{source_schema}.{table_prefix}_otel_metrics"

In [ ]:
# Common timestamp conversion used across all metrics tables.
TS_GAUGE = "TIMESTAMP_MICROS(CAST(gauge.time_unix_nano / 1000 AS BIGINT))"
TS_SUM = "TIMESTAMP_MICROS(CAST(sum.time_unix_nano / 1000 AS BIGINT))"

In [ ]:
@dlt.table(
    comment="Node-level CPU and memory metrics — one row per metric per timestamp",
)
def node_compute():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.node.cpu.%")) | (F.col("name").like("k8s.node.memory.%"))
        )
        .selectExpr(
            """TIMESTAMP_MICROS(CAST(
                CASE
                    WHEN metric_type = 'gauge' THEN gauge.time_unix_nano
                    WHEN metric_type = 'sum'   THEN sum.time_unix_nano
                END / 1000 AS BIGINT
            )) AS timestamp""",
            "resource.attributes['k8s.node.name'] AS node",
            "name AS metric_name",
            "REPLACE(REPLACE(name, 'k8s.node.cpu.', ''), 'k8s.node.memory.', '') AS metric_short_name",
            "CASE WHEN name LIKE '%.cpu.%' THEN 'cpu' ELSE 'memory' END AS metric_domain",
            "metric_type",
            "CASE WHEN metric_type = 'sum' THEN sum.is_monotonic END AS is_monotonic",
            """CASE
                WHEN metric_type = 'gauge' THEN gauge.value
                WHEN metric_type = 'sum'   THEN sum.value
            END AS value""",
            "unit",
        )
        .filter(F.col("timestamp").isNotNull())
    )

In [ ]:
@dlt.table(
    comment="Node-level filesystem metrics — pivoted with available/capacity/usage per timestamp",
)
def node_filesystem():
    return (
        spark.read.table(source_table)
        .filter(
            (F.col("name").like("k8s.node.filesystem.%")) & F.col("gauge.time_unix_nano").isNotNull()
        )
        .selectExpr(
            f"{TS_GAUGE} AS timestamp",
            "resource.attributes['k8s.node.name'] AS node",
            "name",
            "gauge.value AS value",
        )
        .groupBy("timestamp", "node")
        .agg(
            F.max(F.when(F.col("name") == "k8s.node.filesystem.available", F.col("value"))).alias("available_bytes"),
            F.max(F.when(F.col("name") == "k8s.node.filesystem.capacity", F.col("value"))).alias("capacity_bytes"),
            F.max(F.when(F.col("name") == "k8s.node.filesystem.usage", F.col("value"))).alias("usage_bytes"),
        )
        .withColumn(
            "usage_pct",
            F.round(F.col("usage_bytes") * 100.0 / F.when(F.col("capacity_bytes") != 0, F.col("capacity_bytes")), 2),
        )
    )

In [ ]:
@dlt.table(
    comment="Node-level network IO and error metrics with interface and direction",
)
def node_network():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.node.network.%")) & F.col("sum.time_unix_nano").isNotNull()
        )
        .selectExpr(
            f"{TS_SUM} AS timestamp",
            "resource.attributes['k8s.node.name'] AS node",
            "name AS metric_name",
            "REPLACE(name, 'k8s.node.network.', '') AS metric_short_name",
            "sum.attributes['interface'] AS interface",
            "sum.attributes['direction'] AS direction",
            "sum.is_monotonic",
            "sum.value AS value",
            "unit",
        )
    )

In [ ]:
@dlt.table(
    comment="Pod-level CPU and memory metrics with full K8s resource hierarchy",
)
def pod_compute():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.pod.cpu.%")) | (F.col("name").like("k8s.pod.memory.%"))
        )
        .selectExpr(
            """TIMESTAMP_MICROS(CAST(
                CASE
                    WHEN metric_type = 'gauge' THEN gauge.time_unix_nano
                    WHEN metric_type = 'sum'   THEN sum.time_unix_nano
                END / 1000 AS BIGINT
            )) AS timestamp""",
            "resource.attributes['k8s.node.name'] AS node",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.deployment.name'] AS deployment",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "resource.attributes['k8s.container.name'] AS container",
            "resource.attributes['service.name'] AS service",
            "resource.attributes['service.namespace'] AS service_namespace",
            "name AS metric_name",
            "REPLACE(REPLACE(name, 'k8s.pod.cpu.', ''), 'k8s.pod.memory.', '') AS metric_short_name",
            "CASE WHEN name LIKE '%.cpu.%' THEN 'cpu' ELSE 'memory' END AS metric_domain",
            "metric_type",
            "CASE WHEN metric_type = 'sum' THEN sum.is_monotonic END AS is_monotonic",
            """CASE
                WHEN metric_type = 'gauge' THEN gauge.value
                WHEN metric_type = 'sum'   THEN sum.value
            END AS value""",
            "unit",
        )
        .filter(F.col("timestamp").isNotNull())
    )

In [ ]:
@dlt.table(
    comment="Pod-level filesystem metrics — pivoted with available/capacity/usage per timestamp",
)
def pod_filesystem():
    return (
        spark.read.table(source_table)
        .filter(
            (F.col("name").like("k8s.pod.filesystem.%")) & F.col("gauge.time_unix_nano").isNotNull()
        )
        .selectExpr(
            f"{TS_GAUGE} AS timestamp",
            "resource.attributes['k8s.node.name'] AS node",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.deployment.name'] AS deployment",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "resource.attributes['service.name'] AS service",
            "name",
            "gauge.value AS value",
        )
        .groupBy("timestamp", "node", "namespace", "deployment", "pod", "pod_uid", "service")
        .agg(
            F.max(F.when(F.col("name") == "k8s.pod.filesystem.available", F.col("value"))).alias("available_bytes"),
            F.max(F.when(F.col("name") == "k8s.pod.filesystem.capacity", F.col("value"))).alias("capacity_bytes"),
            F.max(F.when(F.col("name") == "k8s.pod.filesystem.usage", F.col("value"))).alias("usage_bytes"),
        )
        .withColumn(
            "usage_pct",
            F.round(F.col("usage_bytes") * 100.0 / F.when(F.col("capacity_bytes") != 0, F.col("capacity_bytes")), 2),
        )
    )

In [ ]:
@dlt.table(
    comment="Pod-level network IO and error metrics with interface and direction",
)
def pod_network():
    return (
        spark.readStream.table(source_table)
        .filter(
            (F.col("name").like("k8s.pod.network.%")) & F.col("sum.time_unix_nano").isNotNull()
        )
        .selectExpr(
            f"{TS_SUM} AS timestamp",
            "resource.attributes['k8s.node.name'] AS node",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.deployment.name'] AS deployment",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "resource.attributes['service.name'] AS service",
            "name AS metric_name",
            "REPLACE(name, 'k8s.pod.network.', '') AS metric_short_name",
            "sum.attributes['interface'] AS interface",
            "sum.attributes['direction'] AS direction",
            "sum.is_monotonic",
            "sum.value AS value",
            "unit",
        )
    )

In [ ]:
@dlt.table(
    comment="Latest CPU/memory requests and limits per container (from k8s_cluster receiver). "
            "Holds one row per container with the most recent observed spec values.",
)
def container_resource_spec():
    from pyspark.sql.window import Window
    base = (
        spark.read.table(source_table)
        .filter(
            (F.col("name") == "k8s.container.cpu_request")
            | (F.col("name") == "k8s.container.cpu_limit")
            | (F.col("name") == "k8s.container.memory_request")
            | (F.col("name") == "k8s.container.memory_limit")
        )
        .filter(F.col("gauge.time_unix_nano").isNotNull())
        .selectExpr(
            f"{TS_GAUGE} AS timestamp",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "COALESCE(gauge.attributes['k8s.container.name'], "
            "         resource.attributes['k8s.container.name']) AS container",
            "resource.attributes['k8s.node.name'] AS node",
            "name AS metric_name",
            "gauge.value AS value",
        )
        .filter(F.col("pod").isNotNull() & F.col("container").isNotNull())
    )
    w = Window.partitionBy("namespace", "pod", "container", "metric_name").orderBy(F.desc("timestamp"))
    latest = (
        base.withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
    return (
        latest.groupBy("namespace", "pod", "pod_uid", "container", "node")
        .agg(
            F.max(F.when(F.col("metric_name") == "k8s.container.cpu_request", F.col("value"))).alias("cpu_request_cores"),
            F.max(F.when(F.col("metric_name") == "k8s.container.cpu_limit", F.col("value"))).alias("cpu_limit_cores"),
            F.max(F.when(F.col("metric_name") == "k8s.container.memory_request", F.col("value"))).alias("memory_request_bytes"),
            F.max(F.when(F.col("metric_name") == "k8s.container.memory_limit", F.col("value"))).alias("memory_limit_bytes"),
            F.max("timestamp").alias("spec_observed_at"),
        )
    )

In [ ]:
@dlt.table(
    comment="Estimated node capacity (CPU cores, memory bytes) derived from "
            "k8s.node.{cpu,memory}.usage divided by *.utilization. Falls back to "
            "max observed usage when utilization unavailable.",
)
def node_capacity_estimated():
    src = (
        spark.read.table(source_table)
        .filter(
            (F.col("name") == "k8s.node.cpu.usage")
            | (F.col("name") == "k8s.node.cpu.utilization")
            | (F.col("name") == "k8s.node.memory.usage")
            | (F.col("name") == "k8s.node.memory.working_set")
        )
        .filter(F.col("gauge.time_unix_nano").isNotNull())
        .selectExpr(
            f"{TS_GAUGE} AS timestamp",
            "resource.attributes['k8s.node.name'] AS node",
            "name",
            "gauge.value AS value",
        )
    )
    pivoted = (
        src.groupBy("timestamp", "node")
        .agg(
            F.max(F.when(F.col("name") == "k8s.node.cpu.usage", F.col("value"))).alias("cpu_usage"),
            F.max(F.when(F.col("name") == "k8s.node.cpu.utilization", F.col("value"))).alias("cpu_util"),
            F.max(F.when(F.col("name") == "k8s.node.memory.usage", F.col("value"))).alias("mem_usage"),
            F.max(F.when(F.col("name") == "k8s.node.memory.working_set", F.col("value"))).alias("mem_working_set"),
        )
    )
    derived = pivoted.selectExpr(
        "node",
        "timestamp",
        "cpu_usage",
        "mem_usage",
        "mem_working_set",
        "CASE WHEN cpu_util IS NOT NULL AND cpu_util > 0.01 THEN cpu_usage / cpu_util END AS cpu_capacity_estimate",
    )
    return (
        derived.groupBy("node")
        .agg(
            F.coalesce(F.max("cpu_capacity_estimate"), F.max("cpu_usage")).alias("cpu_capacity_cores"),
            F.round(F.coalesce(F.max("mem_working_set"), F.max("mem_usage")) * 1.2, 0).alias("memory_capacity_bytes_estimate"),
            F.max("cpu_usage").alias("cpu_usage_peak_cores"),
            F.max("mem_usage").alias("memory_usage_peak_bytes"),
            F.max("timestamp").alias("observed_at"),
        )
    )

In [ ]:
@dlt.table(
    comment="Per-pod bin-packing analysis: requested vs actual CPU/memory usage. "
            "Lower efficiency = more over-provisioning.",
)
def bin_packing_pod_efficiency():
    pod_spec = (
        dlt.read("container_resource_spec")
        .groupBy("namespace", "pod", "pod_uid", "node")
        .agg(
            F.sum("cpu_request_cores").alias("cpu_request_cores"),
            F.sum("cpu_limit_cores").alias("cpu_limit_cores"),
            F.sum("memory_request_bytes").alias("memory_request_bytes"),
            F.sum("memory_limit_bytes").alias("memory_limit_bytes"),
        )
    )
    pod_usage = (
        dlt.read("pod_compute")
        .filter(F.col("metric_short_name") == "usage")
        .groupBy("namespace", "pod", "pod_uid", "node", "deployment", "service", "metric_domain")
        .agg(
            F.expr("percentile_approx(value, 0.95)").alias("p95"),
            F.avg("value").alias("avg"),
            F.max("value").alias("max"),
        )
    )
    pod_usage_pivoted = (
        pod_usage.groupBy("namespace", "pod", "pod_uid", "node", "deployment", "service")
        .agg(
            F.max(F.when(F.col("metric_domain") == "cpu", F.col("p95"))).alias("cpu_p95_cores"),
            F.max(F.when(F.col("metric_domain") == "cpu", F.col("avg"))).alias("cpu_avg_cores"),
            F.max(F.when(F.col("metric_domain") == "cpu", F.col("max"))).alias("cpu_max_cores"),
            F.max(F.when(F.col("metric_domain") == "memory", F.col("p95"))).alias("memory_p95_bytes"),
            F.max(F.when(F.col("metric_domain") == "memory", F.col("avg"))).alias("memory_avg_bytes"),
            F.max(F.when(F.col("metric_domain") == "memory", F.col("max"))).alias("memory_max_bytes"),
        )
    )
    return (
        pod_spec.join(pod_usage_pivoted, ["namespace", "pod", "pod_uid", "node"], "left")
        .withColumn(
            "cpu_efficiency_pct",
            F.when(
                (F.col("cpu_request_cores").isNotNull()) & (F.col("cpu_request_cores") > 0),
                F.round(F.col("cpu_p95_cores") * 100.0 / F.col("cpu_request_cores"), 1),
            ),
        )
        .withColumn(
            "memory_efficiency_pct",
            F.when(
                (F.col("memory_request_bytes").isNotNull()) & (F.col("memory_request_bytes") > 0),
                F.round(F.col("memory_p95_bytes") * 100.0 / F.col("memory_request_bytes"), 1),
            ),
        )
        .withColumn(
            "cpu_waste_cores",
            F.when(
                F.col("cpu_request_cores").isNotNull() & F.col("cpu_p95_cores").isNotNull(),
                F.greatest(F.col("cpu_request_cores") - F.col("cpu_p95_cores"), F.lit(0.0)),
            ),
        )
        .withColumn(
            "memory_waste_bytes",
            F.when(
                F.col("memory_request_bytes").isNotNull() & F.col("memory_p95_bytes").isNotNull(),
                F.greatest(F.col("memory_request_bytes") - F.col("memory_p95_bytes"), F.lit(0.0)),
            ),
        )
    )

In [ ]:
@dlt.table(
    comment="Per-node bin-packing summary. Joins pod-level totals (requests, p95 usage) "
            "with node capacity estimate. Use this to find consolidation candidates.",
)
def bin_packing_node_summary():
    pod_eff = dlt.read("bin_packing_pod_efficiency")
    by_node = (
        pod_eff.groupBy("node")
        .agg(
            F.count("*").alias("pod_count"),
            F.sum("cpu_request_cores").alias("cpu_requested_cores"),
            F.sum("memory_request_bytes").alias("memory_requested_bytes"),
            F.sum("cpu_p95_cores").alias("cpu_used_p95_cores"),
            F.sum("memory_p95_bytes").alias("memory_used_p95_bytes"),
            F.sum("cpu_waste_cores").alias("cpu_waste_cores"),
            F.sum("memory_waste_bytes").alias("memory_waste_bytes"),
        )
    )
    cap = dlt.read("node_capacity_estimated")
    return (
        by_node.join(cap, "node", "left")
        .withColumn(
            "cpu_request_pct_of_capacity",
            F.when(
                F.col("cpu_capacity_cores") > 0,
                F.round(F.col("cpu_requested_cores") * 100.0 / F.col("cpu_capacity_cores"), 1),
            ),
        )
        .withColumn(
            "cpu_usage_pct_of_capacity",
            F.when(
                F.col("cpu_capacity_cores") > 0,
                F.round(F.col("cpu_used_p95_cores") * 100.0 / F.col("cpu_capacity_cores"), 1),
            ),
        )
        .withColumn(
            "memory_request_pct_of_capacity",
            F.when(
                F.col("memory_capacity_bytes_estimate") > 0,
                F.round(F.col("memory_requested_bytes") * 100.0 / F.col("memory_capacity_bytes_estimate"), 1),
            ),
        )
        .withColumn(
            "memory_usage_pct_of_capacity",
            F.when(
                F.col("memory_capacity_bytes_estimate") > 0,
                F.round(F.col("memory_used_p95_bytes") * 100.0 / F.col("memory_capacity_bytes_estimate"), 1),
            ),
        )
    )